In [ ]:
# =====================================
# INSTALAR LIBRERÍAS
# =====================================

!pip -q install gradio transformers sentence-transformers faiss-cpu pypdf accelerate

# =====================================
# IMPORTAR LIBRERÍAS
# =====================================

import gradio as gr
import numpy as np
import faiss

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader


# =====================================
# VARIABLES GLOBALES
# =====================================

documents = []
index = None

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Load tokenizer and model directly for Flan-T5
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
generator_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")


# =====================================
# LEER PDF
# =====================================

def read_pdf(file):

    text = ""

    reader = PdfReader(file)

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text + " "

    # limpiar saltos de línea
    text = text.replace("\n", " ")

    # quitar espacios dobles
    text = " ".join(text.split())

    return text


# =====================================
# CARGAR PDFs
# =====================================

def load_pdfs(files):

    global documents
    global index

    documents = []

    for file in files:

        text = read_pdf(file)

        # Modified: Change how documents are split
        chunks = [text[i:i+500] for i in range(0, len(text), 500)]

        documents.extend(chunks)

    embeddings = embed_model.encode(documents)

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(dimension)

    index.add(np.array(embeddings))

    return "✅ PDFs cargados e indexados correctamente"


# =====================================
# BUSCAR CONTEXTO
# =====================================

def search_docs(question):

    question_embedding = embed_model.encode([question])

    distances, indices = index.search(np.array(question_embedding), 5)

    results = []

    for i in indices[0]:

        results.append(documents[i])

    return " ".join(results)


# =====================================
# CHATBOT
# =====================================

def chat(message, history):

    history.append({"role": "user", "content": message})

    context = search_docs(message)

    prompt = f"""
Utilizando el contexto proporcionado, responde la pregunta de manera clara y completa en español.
No copies el texto literalmente; explica la idea con tus propias palabras.

Contexto:
{context}

Pregunta:
{message}

Respuesta:
"""

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = generator_model.generate(
        **inputs,
        max_length=300,
        min_length=40,
        do_sample=False
    )
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    # Attempt to extract text after "Respuesta:"
    if "Respuesta:" in generated_text:
        answer = generated_text.split("Respuesta:", 1)[1].strip()
    else:
        answer = generated_text # Fallback if "Respuesta:" is not found

    history.append({"role": "assistant", "content": answer})

    return history, history, ""


# =====================================
# INTERFAZ
# =====================================

with gr.Blocks() as demo:

    gr.Markdown("# 🤖 Chatbot PDF Inteligente con FLAN-T5")

    with gr.Row():

        with gr.Column():

            gr.Markdown("### Subir PDFs")

            pdf_files = gr.File(
                file_types=[".pdf"],
                file_count="multiple"
            )

            upload_btn = gr.Button("📂 Cargar PDFs")

            status = gr.Textbox(label="Estado")

        with gr.Column():

            chatbot = gr.Chatbot(
                label="Chatbot",
                type="messages",
                allow_tags=False
            )

            msg = gr.Textbox(label="Pregunta")

            send_btn = gr.Button("Enviar")

    upload_btn.click(
        load_pdfs,
        inputs=pdf_files,
        outputs=status
    )

    send_btn.click(
        chat,
        inputs=[msg, chatbot],
        outputs=[chatbot, chatbot, msg]
    )


demo.launch(share=True)